In [9]:
from tqdm import trange
import numpy as np
import bilby
import numpy as np
from astropy import constants as const
import configparser
import os
import errno
import json
import datetime
import scipy.signal.windows as windows
import matplotlib.pyplot as plt
from bilby.core.utils import infft

"""get current time"""
start = datetime.datetime.now()

"""constants"""
Mo = const.M_sun.value #solar mass [kg]
G = const.G.value #Newton constant [m^3 kg^-1 s^2]
c = const.c.value #light speed [m s^-1]
pc = const.pc.value #1pc [m]

"""get ringdown waveform from config file"""
def check_prior_effeciency(config_file_path):

    config_path = config_file_path
    config_ini = configparser.ConfigParser()
    config_ini.read(config_path)

    config_setting = config_ini['setting']
    config_injection = config_ini['injection_parameters']
    config_wf_args = config_ini['waveform_arguments']
    config_others = config_ini['other_parameters']

    """set the event to be analyzed"""
    event_name = config_setting['event_name']
    outdir_path = config_setting['outdir_path']
    outdir = outdir_path + 'outdir_' + event_name
    label = event_name
    logger = bilby.core.utils.logger
    bilby.core.utils.setup_logger(outdir=outdir, label=label)

    """output to .log file"""
    comment = config_setting['comment']
    bilby.core.utils.logger.info(comment.replace('_', ' '))
    bilby.core.utils.logger.info('event_name : '+event_name)

    one_mode=False
    mode_number = config_setting['mode_number']
    if mode_number=='one_mode':
        one_mode=True
    bilby.core.utils.logger.info('mode_number : '+mode_number)

    """output waveform args"""
    for key in config_wf_args.keys():
        bilby.core.utils.logger.info('{0} : {1}'.format(key, config_wf_args[key]))

        """waveform"""
    def toy_model_of_two_QNMs(time, A, alpha, f1, tau1, f2, tau2, phi1, phi2, geocent_time):
        """
        Returns
        -------
        dict:
            A dictionary containing "plus" and "cross" entries.
        """
        waveform1 = np.zeros(len(time), dtype=complex)
        waveform2 = np.zeros(len(time), dtype=complex)

        A = A * 1e-20
        w1 = (2*np.pi*f1 + 1j / tau1)
        w2 = (2*np.pi*f2 + 1j / tau2)
        delta_w = w2 - w1

        tidx = time >= geocent_time

        waveform1[tidx] = A / delta_w * np.exp(-1j * (w1 * (time[tidx] - geocent_time)) + 1j * phi1)
        waveform2[tidx] = -A * (1 + delta_w * alpha) / delta_w * np.exp(-1j * (w2 * (time[tidx] - geocent_time)) + 1j * phi2)

        if one_mode:
            waveform2 = np.zeros(len(time), dtype=complex)

        # tukey_window = windows.tukey(len(time), 0.1)  # Tukey window with alpha=0.1

        plus = (waveform1 + waveform2).real #* tukey_window
        cross = (waveform1 + waveform2).imag #* tukey_window
        return {"plus": plus, "cross": cross}

    def get_each_overtone(time, A, alpha, f1, tau1, f2, tau2, phi1, phi2, geocent_time):
        """
        Returns
        -------
        dict:
            A dictionary containing "waveform1", "waveform2", and "plus" entries.
        """
        waveform1 = np.zeros(len(time), dtype=complex)
        waveform2 = np.zeros(len(time), dtype=complex)

        A = A * 1e-20
        w1 = (2*np.pi*f1 + 1j / tau1)
        w2 = (2*np.pi*f2 + 1j / tau2)
        delta_w = w2 - w1

        tidx = time >= geocent_time

        waveform1[tidx] = A / delta_w * np.exp(-1j * (w1 * (time[tidx] - geocent_time)) + 1j * phi1)
        waveform2[tidx] = -A * (1 + delta_w * alpha) / delta_w * np.exp(-1j * (w2 * (time[tidx] - geocent_time)) + 1j * phi2)

        if one_mode:
            waveform2 = np.zeros(len(time), dtype=complex)

        plus = (waveform1 + waveform2).real
        return {"waveform1": waveform1, "waveform2": waveform2, "plus": plus}
    
    def damped_sinusoid(time, A1, A2, f1, f2, tau1, tau2, phi1, phi2, geocent_time):
        waveform1 = np.zeros(len(time), dtype=complex)
        waveform2 = np.zeros(len(time), dtype=complex)

        A1 = A1 * 1e-20
        A2 = A2 * 1e-20
        w1 = (2*np.pi*f1 + 1j / tau1)
        w2 = (2*np.pi*f2 + 1j / tau2)

        tidx = time >= geocent_time

        waveform1[tidx] = A1 * np.exp(-1j * (w1 * (time[tidx] - geocent_time)) + 1j * phi1)
        waveform2[tidx] = A2 * np.exp(-1j * (w2 * (time[tidx] - geocent_time)) + 1j * phi2)

        if one_mode:
            waveform2 = np.zeros(len(time), dtype=complex)

        plus = (waveform1 + waveform2).real
        cross = (waveform1 + waveform2).imag

        tukey_window = windows.tukey(len(time), 0.1)  # Tukey window with alpha=0.1
        plus *= tukey_window
        cross *= tukey_window

        return {"plus": plus, "cross": cross}

    def get_each_overtone_as_damped_sinusoid_parameters(time, A1, A2, f1, f2, tau1, tau2, phi1, phi2, geocent_time):
        waveform1 = np.zeros(len(time), dtype=complex)
        waveform2 = np.zeros(len(time), dtype=complex)

        A1 = A1 * 1e-20
        A2 = A2 * 1e-20
        w1 = (2*np.pi*f1 + 1j / tau1)
        w2 = (2*np.pi*f2 + 1j / tau2)

        tidx = time >= geocent_time

        waveform1[tidx] = A1 * np.exp(-1j * (w1 * (time[tidx] - geocent_time)) + 1j * phi1)
        waveform2[tidx] = A2 * np.exp(-1j * (w2 * (time[tidx] - geocent_time)) + 1j * phi2)

        if one_mode:
            waveform2 = np.zeros(len(time), dtype=complex)

        tukey_window = windows.tukey(len(time), 0.1)  # Tukey window with alpha=0.1
        waveform1 *= tukey_window
        waveform2 *= tukey_window

        plus = (waveform1 + waveform2).real * tukey_window
        return {"waveform1": waveform1, "waveform2": waveform2, "plus": plus}

    """set parameters"""
    duration = float(config_others['duration'])
    sampling_frequency = float(config_others['sampling_frequency'])
    trigger_time = float(config_injection['geocent_time'])
    post_trigger_duration = float(config_others['post_trigger_duration'])
    start_time = trigger_time - duration + post_trigger_duration
    bilby.core.utils.logger.info('duration : {}'.format(duration))
    bilby.core.utils.logger.info('sampling frequency : {}'.format(sampling_frequency))
    bilby.core.utils.logger.info('trigger time : {}'.format(trigger_time))
    bilby.core.utils.logger.info('start time : {}'.format(start_time))

    """set injection parameters and waveform generator"""
    inject_as_damped_sinusoid = True
    # inject_as_damped_sinusoid = False
    if inject_as_damped_sinusoid:
        A = float(config_injection['A']) * 1e-20  # Convert to strain unit
        alpha = float(config_injection['alpha'])
        w1 = (2 * np.pi * float(config_injection['f1']) + 1j / float(config_injection['tau1']))
        w2 = (2 * np.pi * float(config_injection['f2']) + 1j / float(config_injection['tau2']))
        delta_w = w2 - w1
        A1_complex = A / delta_w
        A1 = np.abs(A1_complex) / 1e-20
        phi1_from_amp = np.angle(A1_complex)
        phi1 = float(config_injection['phi1']) + phi1_from_amp
        A2_complex = - A * (1 + alpha * delta_w) / delta_w
        A2 = np.abs(A2_complex) / 1e-20
        phi2_from_amp = np.angle(A2_complex)
        phi2 = float(config_injection['phi2']) + phi2_from_amp
        bilby.core.utils.logger.info('A1 : {}'.format(A1))
        bilby.core.utils.logger.info('A2 : {}'.format(A2))
        bilby.core.utils.logger.info('alpha : {}'.format(alpha))
        bilby.core.utils.logger.info('w1 : {}'.format(w1))
        bilby.core.utils.logger.info('w2 : {}'.format(w2))
        bilby.core.utils.logger.info('delta_w : {}'.format(delta_w))
        bilby.core.utils.logger.info('delta_w * alpha : {}'.format(delta_w * alpha))
        bilby.core.utils.logger.info('phi1 : {}'.format(phi1))
        bilby.core.utils.logger.info('phi2 : {}'.format(phi2))
        injection_parameters = dict(
                                    A1 = A1,
                                    A2 = A2,
                                    f1 = float(config_injection['f1']),
                                    tau1 = float(config_injection['tau1']),
                                    f2 = float(config_injection['f2']),
                                    tau2 = float(config_injection['tau2']),
                                    phi1 = phi1,
                                    phi2 = phi2,
                                    ra = float(config_injection['ra']),
                                    dec = float(config_injection['dec']),
                                    psi = float(config_injection['psi']),
                                    geocent_time = float(config_injection['geocent_time']),
                                    )

        waveform_generator = bilby.gw.waveform_generator.WaveformGenerator(
                                duration = duration,
                                sampling_frequency = sampling_frequency,
                                time_domain_source_model = damped_sinusoid,
                                start_time = start_time,
                                )
    else:
        injection_parameters = dict(
                                    A = float(config_injection['A']),
                                    alpha = float(config_injection['alpha']),
                                    f1 = float(config_injection['f1']),
                                    tau1 = float(config_injection['tau1']),
                                    f2 = float(config_injection['f2']),
                                    tau2 = float(config_injection['tau2']),
                                    phi1 = float(config_injection['phi1']),
                                    phi2 = float(config_injection['phi2']),
                                    ra = float(config_injection['ra']),
                                    dec = float(config_injection['dec']),
                                    psi = float(config_injection['psi']),
                                    geocent_time = float(config_injection['geocent_time']),
                                    )

        waveform_generator = bilby.gw.waveform_generator.WaveformGenerator(
                                duration = duration,
                                sampling_frequency = sampling_frequency,
                                time_domain_source_model = toy_model_of_two_QNMs,
                                start_time = start_time,
                                )
    
    """set interferometers"""
    ifos = bilby.gw.detector.InterferometerList(['H1', 'L1', 'V1'])
    #ifos.set_strain_data_from_zero_noise(
    #                                sampling_frequency = sampling_frequency,
    #                                duration = duration,
    #                                start_time = start_time
    #                                )

    ifos.set_strain_data_from_power_spectral_densities(
                                    sampling_frequency = sampling_frequency,
                                    duration = duration,
                                    start_time = start_time
                                    )
    for interferometer in ifos:
        interferometer.minimum_frequency = float(config_wf_args['minimum_frequency'])
        interferometer.maximum_frequency = float(config_wf_args['maximum_frequency'])
    """inject signal"""
    ifos.inject_signal(
        waveform_generator=waveform_generator,
        parameters=injection_parameters,
        raise_error=False
    )

    priors = injection_parameters.copy()
    # priors['A1'] = bilby.core.prior.LogUniform(1e-2, 1e2, r"$A_1$")
    priors['A1'] = bilby.core.prior.Uniform(0, 100, r"$A_1$")
    # priors['A2'] = bilby.core.prior.LogUniform(1e-2, 1e2, r"$A_2$")
    priors['A2'] = bilby.core.prior.Uniform(0, 100, r"$A_2$")

    priors['f1'] = bilby.core.prior.Uniform(20, 500, r"$f_1$ [Hz]")
    priors['f2'] = bilby.core.prior.Uniform(20, 500, r"$f_2$ [Hz]")
    
    priors['tau1'] = bilby.core.prior.Uniform(-0.05, -0.0005, r"$\tau_1$ [ms]")
    priors['tau2'] = bilby.core.prior.Uniform(-0.05, -0.0005, r"$\tau_2$ [ms]")
    
    priors['phi1'] = bilby.core.prior.Uniform(-np.pi, np.pi, r"$\phi_1$ [rad]", boundary='periodic')
    priors['phi2'] = bilby.core.prior.Uniform(-np.pi, np.pi, r"$\phi_2$ [rad]", boundary='periodic')

    priors['ra'] = bilby.core.prior.Uniform(0, 2*np.pi, name='ra', latex_label='$\\alpha$', unit='rad', boundary='periodic')
    priors['dec'] = bilby.core.prior.Cosine(name='dec', latex_label='$\\delta$', unit='rad', boundary='reflective')
    priors['psi'] = bilby.core.prior.Uniform(0, np.pi, name='psi', latex_label='$\\psi$', unit='rad', boundary='periodic')
    priors['geocent_time'] = bilby.core.prior.Uniform(minimum = trigger_time - duration/2, maximum = trigger_time + duration/2)

    if 'fix_parameters' in config_ini.keys():
        fix_list =  json.loads(config_ini['fix_parameters']['fix_list']) #str to list
        for key in fix_list:
            if key == 'A2':
                if one_mode:
                    priors[key] = 0
                else:
                    priors[key] = injection_parameters[key]
            else:
                priors[key] = injection_parameters[key]

    likelihood = bilby.gw.likelihood.GravitationalWaveTransient(
                                                                interferometers=ifos,
                                                                priors=priors,
                                                                waveform_generator=waveform_generator,
                                                                distance_marginalization=False,
                                                                phase_marginalization=False
                                                                )


    n_total = 1000
    n_good = 0
    for _ in trange(n_total):
        for key in priors:
            if key not in fix_list:
                sample = priors[key].sample()
            logL = likelihood.log_likelihood(sample)
            if np.isfinite(logL):
                n_good += 1
    print("Prior sampling efficiency:", n_good / n_total)

In [10]:
# config_path = '/Users/hayato/research/ringdown/injection_analysis/config/test_one_mode.ini'
# config_path = '/Users/hayato/research/ringdown/injection_analysis/config/test_two_mode.ini'
# config_path = '/Users/hayato/research/ringdown/injection_analysis/config/test_two_mode_shift_wreal.ini'
# config_path = '/Users/hayato/research/ringdown/injection_analysis/config/test_two_mode_shift_wimag.ini'
config_path = '/Users/hayato/research/ringdown/injection_analysis/config/shiftRe_to_220_dw0.1w1_snr100.ini'
# config_path = '/Users/hayato/research/ringdown/injection_analysis/config/shiftRe_to_220_dw0.01w1_snr100.ini'
# config_path = '/Users/hayato/research/ringdown/injection_analysis/config/shiftRe_to_220_dw0.001w1_snr100.ini'
# config_path = '/Users/hayato/research/ringdown/injection_analysis/config/shiftIm_to_220_dw0.1w1_snr100.ini'
# config_path = '/Users/hayato/research/ringdown/injection_analysis/config/shiftIm_to_220_dw0.01w1_snr100.ini'
# config_path = '/Users/hayato/research/ringdown/injection_analysis/config/shiftIm_to_220_dw0.001w1_snr100.ini'
# config_path = '/Users/hayato/research/ringdown/injection_analysis/config/shiftRe_to_220_dw0.2w1_snr100_change_alpha.ini'

check_prior_effeciency(config_path)

13:01 bilby INFO    : test, just two damped sinusoid
13:01 bilby INFO    : event_name : test_two_mode
13:01 bilby INFO    : mode_number : two_mode
13:01 bilby INFO    : minimum_frequency : 20
13:01 bilby INFO    : maximum_frequency : 512
13:01 bilby INFO    : duration : 0.5
13:01 bilby INFO    : sampling frequency : 4096.0
13:01 bilby INFO    : trigger time : 0.0
13:01 bilby INFO    : start time : -0.25
13:01 bilby INFO    : A1 : 0.6255029111994458
13:01 bilby INFO    : A2 : 1.2510058223988911
13:01 bilby INFO    : alpha : -0.007693762745380632
13:01 bilby INFO    : w1 : (1264.4143462793452-301.02689574732483j)
13:01 bilby INFO    : w2 : (1134.4389320257144-301.02689574732483j)
13:01 bilby INFO    : delta_w : (-129.9754142536308+0j)
13:01 bilby INFO    : delta_w * alpha : (0.9999999999999993-0j)
13:01 bilby INFO    : phi1 : -3.141592653589793
13:01 bilby INFO    : phi2 : 0.0
13:01 bilby INFO    : Waveform generator initiated with
  frequency_domain_source_model: None
  time_domain_sour

TypeError: GravitationalWaveTransient.log_likelihood() takes 1 positional argument but 2 were given